# Counsel: AI Business Consultant

Upload a dataset, state a business goal, and get evidence-based recommendations, a dashboard, and Power BI-ready tables.

**The core idea:** the LLM plans and writes, while code executes every calculation and verifies every number it cites. Recommendations stay grounded in the data instead of in the model's memory.

## How it works

| Stage | The LLM does | The code does |
|---|---|---|
| Data prep | Decides each column's role (metric, dimension, sensitive...) and how to handle nulls | Applies the plan and refuses unsafe decisions (e.g. filling a column that is 80% empty) |
| Planning | Chooses which analysis tools answer the question | Blocks any plan that touches personal or free-text columns |
| Execution | Nothing | Runs the analysis with pandas and summarizes it (e.g. flags negligible gaps between groups) |
| Consulting | Writes a structured report: situation, findings, prioritized recommendations | Verifies that every number cited as evidence exists in the results |

## Design decisions

- **General, not customized:** no column names are hardcoded. The LLM reads each dataset and decides what its columns mean.
- **The LLM never calculates:** every figure comes from code.
- **Nulls are not always errors:** a missing return reason usually means "not returned", so columns are never dropped just for having nulls.
- **Plans are cached:** the same dataset schema or question does not pay for the same plan twice.
- **Quality is measured:** a fixed set of questions (including ones that must be refused) is re-run after every change.

## Notebook map

1. **Setup:** environment, LLMs, imports
2. **Data prep:** load the file, let the LLM plan, apply the plan with guards
3. **Pipeline:** planner, executor, summaries
4. **Consultant:** the agent that writes and verifies the report
5. **Test questions:** example runs
6. **Export:** Power BI tables and the HTML dashboard
7. **Evaluation:** fixed questions, pass/fail table

## Setup

- Put your Groq API key in a local `.env` file (never commit it). The notebook reads `GROQ_API_KEY_REASONING`, `GROQ_API_KEY_RESPONDING`, `GROQ_API_KEY_PREPARING` and `GROQ_API_KEY_CONSULTING`; they can all hold the same key.
- Place your dataset in `data/` (ignored by git).
- The Streamlit app (`app.py`) wraps the same pipeline; each user brings their own key.

## Known limitations

- If the LLM's data-prep plan fails completely, the run continues with every column marked `other`, and personal columns are not hidden.
- Numbers in the key findings are verified in the exports and dashboard, not while the report is generated.
- KPI cards show totals even for columns where a sum has no meaning (e.g. age).
- Column statistics, including a few example values per text column, are sent to the LLM provider during data prep.

# 01 Environment Setup and Imports

In [1]:
%pip install groq python-dotenv langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq

## 02 LLMs Setup

In [3]:
load_dotenv(find_dotenv(usecwd=True))

reasoning_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY_REASONING"),
    temperature=0.5,
    reasoning_effort="high",
)

responding_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY_RESPONDING"),
    temperature=0.3,
)

# Data prep: a structured "plan" task, so medium reasoning is enough
prep_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY_PREPARING"),
    temperature=0.2,
    reasoning_effort="medium",
)

consultant_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY_CONSULTING"),
    temperature=0.3,
    reasoning_effort="medium",
)

### importing all tools from `src`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

SRC = ROOT / "src"
sys.path.append(str(SRC))                     # the flat files (loader.py, data_prep.py, ...)
for folder in SRC.iterdir():                  # plus subfolders (agents/)
    if folder.is_dir() and not folder.name.startswith(("_", ".")):
        sys.path.append(str(folder))

from loader import load_file
from data_prep import plan_data_prep, apply_prep_plan, prep_to_text
from analysis import (
    descriptive_analysis,
    group_analysis,
    ratio_analysis,
    time_series_analysis,
    correlation_analysis,
    detect_anomalies,
)
from planner import plan_analysis, PlannerError, PlanStep, AnalysisPlan
from consultant import consult, format_report
from powerbi_export import RunRecord, export_runs

print("ROOT:", ROOT)
print("Imports OK ✅")

ROOT: c:\Users\Viola Ibrahim\Analytix AI\analytix-ai
Imports OK ✅


In [ ]:
df_raw = load_file(str(ROOT / "data" / "ecommerce_sales_customer_analytics_150k.csv")) # 1. loader.py

prep_plan = plan_data_prep(df_raw, prep_llm)
df, prep_log = apply_prep_plan(df_raw, prep_plan)
sensitive = [c.column for c in prep_plan.columns if c.role in ("sensitive", "free_text")]

print("Summary:", prep_plan.dataset_summary)
print("Shape before/after:", df_raw.shape, "/", df.shape)
print("Duplicates removed:", prep_log["duplicates_removed"])
print("Empty columns dropped:", prep_log["empty_columns_dropped"])
print("Dates converted:", prep_log["dates_converted"], "| refused:", prep_log["dates_refused"])

print("\nROLES:")
for c in prep_plan.columns:
    print(f"  {c.column:28s} {c.role:11s} {c.reason}")

print("\nNULL HANDLING:")
for a in prep_log["actions"]:
    print("  ", a)

print("\nREFUSED by code guards:")
for r in prep_log["refused"]:
    print("  ", r)

print("\nRELATIONSHIPS:")
for r in prep_plan.relationships:
    print("  ", r.columns, "->", r.description)

[data prep] attempt 1/4 failed (no structured output), retrying...
Summary: The dataset records individual e‑commerce orders with customer, payment, shipping, product, and marketing details for analytical reporting.
Shape before/after: (138116, 46) / (138116, 46)
Duplicates removed: 0
Empty columns dropped: []
Dates converted: ['order_date', 'order_time'] | refused: []

ROLES:
  order_id                     identifier  Unique order identifier, no missing values
  order_date                   date        Date of the order, essential for time analysis
  order_time                   date        Time of the order, used with date for timestamp
  order_status                 dimension   Order lifecycle status, categorical
  sales_channel                dimension   Channel through which the sale occurred
  customer_id                  identifier  Unique customer identifier
  customer_name                sensitive   Personal name, considered sensitive personal data
  customer_age              

In [8]:
profile_text = prep_to_text(prep_plan, df)
print(profile_text)

Summary: The dataset records individual e‑commerce orders with customer, payment, shipping, product, and marketing details for analytical reporting.
Rows: 138116, Columns: 46
Date columns: ['order_date', 'order_time']
Metric columns (numbers to sum/average): ['customer_age', 'delivery_days', 'estimated_delivery_days', 'customer_rating', 'loyalty_points_earned', 'loyalty_points_redeemed', 'quantity', 'gross_sales', 'discount_amount', 'tax_amount', 'shipping_cost', 'net_sales', 'product_cost', 'profit', 'profit_margin_percentage', 'customer_lifetime_value', 'is_repeat_customer', 'customer_order_count']
Dimension columns (categories to group by): ['order_status', 'sales_channel', 'gender', 'customer_segment', 'customer_type', 'customer_city', 'customer_state', 'customer_country', 'region', 'payment_method', 'payment_status', 'currency', 'shipping_method', 'warehouse', 'delivery_status', 'return_status', 'return_reason', 'review_sentiment', 'marketing_channel', 'campaign_name', 'coupon_cod

In [9]:
# Planner model (the planning logic lives in src/planner/planner.py)
planner_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY_REASONING"),
    temperature=0.3,
    reasoning_effort="medium",
)

In [10]:
# ====== Executor: runs the plan using your real Phase 1 functions ======

def run_step(step: PlanStep, df):
    """Run ONE plan step and return its result."""
    if step.tool == "descriptive_analysis":
        return descriptive_analysis(df, step.column)

    if step.tool == "group_analysis":
        return group_analysis(df, step.metric, step.group_by, step.agg or "sum")

    if step.tool == "ratio_analysis":
        return ratio_analysis(df, step.numerator, step.denominator, step.group_by)

    if step.tool == "time_series_analysis":
        return time_series_analysis(df, step.date_column, step.metric, freq="ME")

    if step.tool == "correlation_analysis":
        return correlation_analysis(df)

    if step.tool == "detect_anomalies":
        return detect_anomalies(df, step.column)

    raise ValueError(f"Unknown tool: {step.tool}")


def execute_plan(plan: AnalysisPlan, df) -> list:
    """Run all steps in order. A failed step is recorded instead of crashing everything."""
    results = []

    if not plan.can_answer:
        print("Cannot answer:", plan.reason_if_not)
        return results

    for step in plan.steps:
        try:
            output = run_step(step, df)
            results.append({"step": step, "output": output, "error": None})
        except Exception as e:
            results.append({"step": step, "output": None, "error": str(e)})

    return results


In [11]:
from langchain_core.prompts import ChatPromptTemplate

# ====== 2) Insight prompt: explain ONLY what the numbers show ======
insight_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a data analyst explaining results to a non-technical user.\n"
     "Rules:\n"
     "- Use ONLY the numbers in the analysis results below. Never invent numbers.\n"
     "- Answer the user's question directly first, then add 2-3 short key insights.\n"
     "- If a step has an ERROR, mention it briefly instead of guessing.\n"
     "- Keep it short and clear.\n"
     "- Answer in English only.\n\n"
     "Analysis results:\n{results}"),
    ("human", "{question}"),
])

# ====== 3) Insight chain: prompt -> responding model ======
insight_chain = insight_prompt | responding_llm


In [12]:
import numpy as np
import pandas as pd


# ====== Summarize each tool output in code before sending it to the LLM ======
def summarize_output(step, output, data) -> str:
    """Turn a raw tool output into a short, unambiguous text summary."""

    # Correlation: send only the top 10 strongest pairs, not the full matrix
    if step.tool == "correlation_analysis":
        id_cols = [c for c in output.columns if str(c).lower().endswith("id")]
        corr = output.drop(index=id_cols, columns=id_cols, errors="ignore")
        mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)  # upper triangle only
        pairs = corr.where(mask).stack()
        pairs = pairs.reindex(pairs.abs().sort_values(ascending=False).index).head(10)
        lines = [f"{a} <-> {b}: {v:.3f}" for (a, b), v in pairs.items()]
        return "Top 10 strongest correlations (by absolute value):\n" + "\n".join(lines)

    # Anomalies: the returned rows ARE the outliers, so say it explicitly
    if step.tool == "detect_anomalies":
        col = step.column
        n_flagged = len(output)
        if n_flagged == 0:
            return f"No anomalies were detected in column '{col}'."
        return (
            f"{n_flagged} anomalous rows (outliers) were detected in '{col}' "
            f"out of {len(data)} total rows.\n"
            f"The outlier values range from {output[col].min()} to {output[col].max()}.\n"
            f"For comparison, the whole column ranges from {data[col].min()} to {data[col].max()} "
            f"with a median of {data[col].median()}."
        )

    # Group / ratio analysis: let the CODE decide if the gap between groups is meaningful
    if step.tool in ("group_analysis", "ratio_analysis") and isinstance(output, pd.Series) and len(output) > 1:
        text = output.to_string()
        lo, hi = float(output.min()), float(output.max())
        if lo > 0:
            gap = (hi - lo) / lo * 100
            verdict = (
                "NEGLIGIBLE (under 5%): treat the groups as practically equal and do NOT rank them."
                if gap < 5 else "meaningful."
            )
            text += (
                f"\nGap between the highest group ({output.idxmax()}) and the lowest group "
                f"({output.idxmin()}): {gap:.1f}% -> {verdict}"
            )
        return text

    # Everything else: Series / DataFrame / dict -> text
    return output.to_string() if hasattr(output, "to_string") else str(output)

def results_to_text(results: list) -> str:
    """Build the text block that is sent to the insight model."""
    parts = []
    for r in results:
        step = r["step"]
        parts.append(f"Step {step.step_id} ({step.tool}) - Goal: {step.goal}")
        if r["error"]:
            parts.append(f"ERROR: {r['error']}")
        else:
            parts.append(summarize_output(step, r["output"], df))
        parts.append("")
    return "\n".join(parts)


# ====== 3) Guard against empty model responses ======
def generate_insights(question: str, results: list) -> str:
    response = insight_chain.invoke({
        "results": results_to_text(results),
        "question": question,
    })
    if not response.content or not response.content.strip():
        return "(The model returned an empty response. Try running the question again.)"
    return response.content


## Consultant Agent

The agent itself lives in `src/agents/consultant.py`. The notebook only wires it into the pipeline.

In [13]:
# ====== Full pipeline: question + goal -> plan -> execute -> consult ======
if "RUNS" not in globals():   
    RUNS = []

def ask_consultant(question: str, goal: str):
    try:
        plan = plan_analysis(f"{question}\n(Business goal: {goal})", profile_text, planner_llm, sensitive)
    except PlannerError as e:
        print(e)
        return None

    if not plan.can_answer:
        print(f"I can't answer this with the current dataset: {plan.reason_if_not}")
        return None

    results = execute_plan(plan, df)

    failed = [r for r in results if r["error"]]
    ok_results = [r for r in results if not r["error"]]

    if failed:
        print("Failed steps:")
        for r in failed:
            print(f"  step {r['step'].step_id} ({r['step'].tool}): {r['error']}")
        print()

    if not ok_results:
        print("No analysis step succeeded, so no business report was generated.")
        return None

    results_text = results_to_text(ok_results)

    try:
        report, verified = consult(question, goal, results_text, consultant_llm)
    except Exception as e:  # BadRequestError / RateLimitError after all retries
        print(f"Structured report failed ({type(e).__name__}), falling back to plain-text insights:\n")
        print(generate_insights(question, ok_results))
        return None

    RUNS.append(RunRecord(question=question, goal=goal, results=ok_results,
                      results_text=results_text, report=report, verified=verified))

    print(format_report(report, verified))
    return report, verified

In [14]:
_ = ask_consultant(
    question="Which regions generate the most net_sales and profit, and what is each region's profit margin?",
    goal="Increase overall profitability",
)

SITUATION: South leads both in net_sales and profit, while North achieves the best profit margin; East has the lowest margin, indicating margin‑growth opportunities.

KEY FINDINGS:
 - South has the highest net_sales at 55,587,994.98 (Step 1).
 - South also has the highest profit at 23,343,946.17 (Step 2).
 - Profit margins by region are: North 0.4610, West 0.4502, South 0.4199, Central 0.4190, East 0.4132 (Step 3).

RECOMMENDATIONS:
[High] Allocate additional marketing and sales resources to the South region to expand market share.
    Evidence (step 1): South net_sales 55,587,994.98; South profit 23,343,946.17   (numbers found in results)
    Impact: Building on the region that already generates the most sales and profit should lift overall profitability.

[Medium] Run a margin‑improvement pilot in the East region focusing on cost reduction and pricing optimization.
    Evidence (step 3): East profit margin 0.4132; North profit margin 0.4610 (gap 11.6%)   (numbers found in results)
  

## Test

In [15]:
_ = ask_consultant(
    question="Which product categories generate the most sales, and how does their profit margin compare?",
    goal="Increase overall profitability",
)


I can't answer this with the current dataset: The dataset description does not include a product category column, so we cannot determine sales or profit margin by product category.


In [16]:
_ = ask_consultant(
    question="Which regions generate the most net_sales and the most profit?",
    goal="Increase overall profitability",
)

print("=" * 70)

SITUATION: South leads both in total net_sales and profit, while North achieves the highest profit margin despite lower sales volume. This mix indicates opportunities to boost profitability by focusing on high‑margin practices and leveraging South's sales strength.

KEY FINDINGS:
 - South region generates the highest net_sales at 55,587,994.98 (Step 1).
 - South region also generates the highest profit at 23,343,946.17 (Step 2).
 - North region has the lowest net_sales (22,565,177.24) and profit (10,402,961.34) among all regions (Steps 1 and 2).

RECOMMENDATIONS:
[High] Develop a targeted campaign in the South region to upsell higher‑margin products and cross‑sell to existing customers.
    Evidence (step 1): South net_sales 55,587,994.98 (Step 1); South profit 23,343,946.17 (Step 2)   (numbers found in results)
    Impact: Leveraging the strong sales volume in South could boost overall profitability if high‑margin products are emphasized.

[Medium] Analyze North's operational and pric

In [17]:
_ = ask_consultant(
    question="Which marketing channels bring the highest net_sales, and how do their average profit compare?",
    goal="Decide where to invest the marketing budget",
)

print("=" * 70)

SITUATION: Organic Search dominates revenue generation, while average profit is virtually uniform across channels. To boost overall performance, focus on leveraging high‑revenue channels, but first gather cost data to ensure profitable spend allocation.

KEY FINDINGS:
 - Organic Search generates the highest net_sales at 35,658,209.46, far exceeding the lowest channel YouTube at 5,197,990.67 (gap 586.0% from Step 1).
 - Average profit per channel is practically the same across all channels, ranging from 555.718156 (Affiliate) to 549.058787 (YouTube), a 1.2% gap considered NEGLIGIBLE (Step 2).

RECOMMENDATIONS:
[High] Run a controlled test reallocating a portion of spend from low‑net_sales channels (e.g., YouTube, TikTok) to Organic Search and Google Ads, then measure revenue and profit outcomes.
    Evidence (step 1): Organic Search net_sales = 35,658,209.46 (Step 1) and Google Ads net_sales = 26,540,586.44 (Step 1)   (numbers found in results)
    Impact: Shifting budget toward the top

In [18]:
_ = ask_consultant(
    question="How did net_sales change over time?",
    goal="Understand sales trends and plan ahead",
)


SITUATION: Net_sales exhibit a clear seasonal pattern with strong November‑December peaks and February troughs, and the high‑season peaks have modestly increased over the five‑year period.

KEY FINDINGS:
 - Net_sales peak dramatically in November and December each year, reaching over 4.2 million in Nov and over 4.5 million in Dec by 2025.
 - The lowest net_sales each year occur in February, falling to around 2.0 million in multiple years.
 - Peak November/December values have risen over the five-year span, from 4.23M/4.90M in 2021 to 4.27M/4.55M in 2025, indicating a growing high-season demand.

RECOMMENDATIONS:
[High] Design and run a targeted promotional campaign or discount program in February to test its effect on sales.
    Evidence (step 1): 2021-02-28 1905413.15; 2022-02-28 2029412.22; 2023-02-28 2035382.66; 2024-02-29 2119755.19; 2025-02-28 2018064.34   (numbers found in results)
    Impact: Boosting sales during the consistently low‑sales February period could raise overall an

In [19]:
metric_cols = [c.column for c in prep_plan.columns if c.role == "metric"]
export_runs(RUNS, ROOT / "outputs" / "powerbi", df=df, metric_columns=metric_cols)

runs.csv: 4 rows
findings.csv: 11 rows
recommendations.csv: 12 rows
analysis_results.csv: 120 rows
kpis.csv: 18 rows


{'runs': WindowsPath('c:/Users/Viola Ibrahim/Analytix AI/analytix-ai/outputs/powerbi/runs.csv'),
 'findings': WindowsPath('c:/Users/Viola Ibrahim/Analytix AI/analytix-ai/outputs/powerbi/findings.csv'),
 'recommendations': WindowsPath('c:/Users/Viola Ibrahim/Analytix AI/analytix-ai/outputs/powerbi/recommendations.csv'),
 'analysis_results': WindowsPath('c:/Users/Viola Ibrahim/Analytix AI/analytix-ai/outputs/powerbi/analysis_results.csv'),
 'kpis': WindowsPath('c:/Users/Viola Ibrahim/Analytix AI/analytix-ai/outputs/powerbi/kpis.csv')}

In [20]:
from dashboard import build_dashboard

metric_cols = [c.column for c in prep_plan.columns if c.role == "metric"]
build_dashboard(RUNS, ROOT / "outputs" / "dashboard.html", df=df, metric_columns=metric_cols)

Dashboard saved: c:\Users\Viola Ibrahim\Analytix AI\analytix-ai\outputs\dashboard.html


WindowsPath('c:/Users/Viola Ibrahim/Analytix AI/analytix-ai/outputs/dashboard.html')

In [21]:
from evaluation import run_evaluation, load_cases

sensitive = [c.column for c in prep_plan.columns if c.role in ("sensitive", "free_text")]

results = run_evaluation(
    df_columns=df.columns,
    plan_fn=lambda q: plan_analysis(q, profile_text, planner_llm, sensitive),
    execute_fn=lambda plan: execute_plan(plan, df),
    results_to_text_fn=results_to_text,
    consult_fn=lambda q, g, t: consult(q, g, t, consultant_llm),
    cases=load_cases(ROOT / "tests" / "eval_cases_ecommerce.json"),
    sensitive_columns=sensitive,
)

[1/8] regions_margin ...
[2/8] sales_channel_margin ...
[3/8] marketing_negligible ...
[4/8] trend_over_time ...
[5/8] delivery_vs_rating ...
[6/8] missing_product_category ...
[7/8] missing_churn ...
[8/8] sensitive_gender ...

                      id  passed  status                                                       tools_used  recs_unverified  findings_unverified  seconds
          regions_margin    True  report                   group_analysis, group_analysis, ratio_analysis                0                    0     45.5
    sales_channel_margin    True  report                   ratio_analysis, group_analysis, group_analysis                1                    0     22.0
    marketing_negligible    True  report                                   group_analysis, group_analysis                0                    0     15.3
         trend_over_time    True  report                                             time_series_analysis                2                    2    100.4
      